In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

In [2]:
np.random.seed(42)
base_temp = 7.8
amplitude = 12
city = "Івано-Франківськ"

In [5]:
np.random.seed(42)

base_temp = 7.8
amplitude = 12
city = "Івано-Франківськ"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"


rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)

        diff = temp - base_temp

        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"

        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": temp,
            "сезон": season(month),
            "відхилення_від_норми": norm_cat
        })


climate = pd.DataFrame(rows)

print(climate)
print("\nКількість рядків:", len(climate))

               місто   рік  місяць  температура  сезон відхилення_від_норми
0   Івано-Франківськ  2021       1         -3.7   зима            холодніше
1   Івано-Франківськ  2021       2         -2.7   зима            холодніше
2   Івано-Франківськ  2021       3          2.4  весна            холодніше
3   Івано-Франківськ  2021       4          9.3  весна             звичайно
4   Івано-Франківськ  2021       5         13.6  весна              тепліше
5   Івано-Франківськ  2021       6         18.0   літо              тепліше
6   Івано-Франківськ  2021       7         21.4   літо              тепліше
7   Івано-Франківськ  2021       8         19.0   літо              тепліше
8   Івано-Франківськ  2021       9         13.3  осінь              тепліше
9   Івано-Франківськ  2021      10          8.3  осінь             звичайно
10  Івано-Франківськ  2021      11          1.3  осінь            холодніше
11  Івано-Франківськ  2021      12         -3.1   зима            холодніше
12  Івано-Фр

In [6]:
table = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"]
)

print("Таблиця спряженості:")
print(table)

Таблиця спряженості:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                        4        4          4
зима                         0        0         12
літо                         0       12          0
осінь                        4        4          4


In [7]:
table_normalized = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"],
    normalize="index"
)

print("Нормована таблиця спряженості:")
print(table_normalized.round(3))

Нормована таблиця спряженості:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                    0.333    0.333      0.333
зима                     0.000    0.000      1.000
літо                     0.000    1.000      0.000
осінь                    0.333    0.333      0.333


In [8]:
chi2, pvalue, dof, expected = chi2_contingency(table)

print(f"Статистика chi2: {chi2:.4f}")
print(f"p-value: {pvalue:.6f}")
print(f"Ступені свободи: {dof}")

expected_table = pd.DataFrame(
    expected,
    index=table.index,
    columns=table.columns
)

print("\nОчікувані частоти:")
print(expected_table.round(2))

Статистика chi2: 38.4000
p-value: 0.000001
Ступені свободи: 6

Очікувані частоти:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      5.0        5.0
зима                       2.0      5.0        5.0
літо                       2.0      5.0        5.0
осінь                      2.0      5.0        5.0


In [9]:
difference = abs(table - expected_table)

print("Абсолютні розбіжності:")
print(difference.round(2))

max_cell = difference.stack().idxmax()
max_difference = difference.stack().max()

print(f"\nНайбільша розбіжність у клітинці: {max_cell[0]} — {max_cell[1]}")
print(f"Величина розбіжності: {max_difference:.2f}")

Абсолютні розбіжності:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      1.0        1.0
зима                       2.0      5.0        7.0
літо                       2.0      7.0        5.0
осінь                      2.0      1.0        1.0

Найбільша розбіжність у клітинці: зима — холодніше
Величина розбіжності: 7.00


In [10]:
alpha = 0.05

if pvalue < alpha:
    print("\nВисновок: H0 відхиляємо. Між сезоном і відхиленням температури є статистично значущий зв'язок.")
else:
    print("\nВисновок: немає достатніх підстав відхиляти H0. Статистично значущого зв'язку не виявлено.")


Висновок: H0 відхиляємо. Між сезоном і відхиленням температури є статистично значущий зв'язок.


In [11]:
print("Очікувані частоти:")
print(expected_table.round(2))

print("\nЧи всі очікувані частоти >= 5?")
print((expected_table >= 5).all().all())

print("\nКлітинки з очікуваною частотою менше 5:")
print(expected_table[expected_table < 5])

Очікувані частоти:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      5.0        5.0
зима                       2.0      5.0        5.0
літо                       2.0      5.0        5.0
осінь                      2.0      5.0        5.0

Чи всі очікувані частоти >= 5?
False

Клітинки з очікуваною частотою менше 5:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      NaN        NaN
зима                       2.0      NaN        NaN
літо                       2.0      NaN        NaN
осінь                      2.0      NaN        NaN


In [12]:
for row in expected_table.index:
    for col in expected_table.columns:
        value = expected_table.loc[row, col]
        if value < 5:
            print(f"{row} — {col}: очікувана частота = {value:.2f}")

весна — звичайно: очікувана частота = 2.00
зима — звичайно: очікувана частота = 2.00
літо — звичайно: очікувана частота = 2.00
осінь — звичайно: очікувана частота = 2.00


In [13]:
climate["відхилення_об'єднане"] = climate["відхилення_від_норми"].replace({
    "холодніше": "не тепліше",
    "звичайно": "не тепліше",
    "тепліше": "тепліше"
})

table_merged = pd.crosstab(
    climate["сезон"],
    climate["відхилення_об'єднане"]
)

print("Таблиця після об'єднання категорій:")
print(table_merged)

Таблиця після об'єднання категорій:
відхилення_об'єднане  не тепліше  тепліше
сезон                                    
весна                          8        4
зима                          12        0
літо                           0       12
осінь                          8        4


In [14]:
chi2_merged, pvalue_merged, dof_merged, expected_merged = chi2_contingency(table_merged)

expected_merged_table = pd.DataFrame(
    expected_merged,
    index=table_merged.index,
    columns=table_merged.columns
)

print(f"chi2 = {chi2_merged:.4f}")
print(f"p-value = {pvalue_merged:.6f}")
print(f"dof = {dof_merged}")

print("\nОчікувані частоти:")
print(expected_merged_table.round(2))

chi2 = 26.0571
p-value = 0.000009
dof = 3

Очікувані частоти:
відхилення_об'єднане  не тепліше  тепліше
сезон                                    
весна                        7.0      5.0
зима                         7.0      5.0
літо                         7.0      5.0
осінь                        7.0      5.0


In [15]:
season_counts = climate["сезон"].value_counts()

season_order = ["зима", "весна", "літо", "осінь"]
season_counts = season_counts.reindex(season_order)

print("Кількість спостережень за сезонами:")
print(season_counts)

Кількість спостережень за сезонами:
сезон
зима     12
весна    12
літо     12
осінь    12
Name: count, dtype: int64


In [16]:
n = len(climate)

expected_seasons = [0.25 * n] * 4

season_result = chisquare(
    f_obs=season_counts,
    f_exp=expected_seasons
)

print(f"Статистика chi2: {season_result.statistic:.4f}")
print(f"p-value: {season_result.pvalue:.6f}")

Статистика chi2: 0.0000
p-value: 1.000000


In [17]:
alpha = 0.05

if season_result.pvalue < alpha:
    print("\nВисновок: H0 відхиляємо. Розподіл спостережень за сезонами статистично відрізняється від рівномірного.")
else:
    print("\nВисновок: немає достатніх підстав відхиляти H0. Розподіл спостережень за сезонами можна вважати рівномірним.")


Висновок: немає достатніх підстав відхиляти H0. Розподіл спостережень за сезонами можна вважати рівномірним.


In [18]:
norm_order = ["холодніше", "звичайно", "тепліше"]

observed_norm = (
    climate["відхилення_від_норми"]
    .value_counts()
    .reindex(norm_order, fill_value=0)
)

print("Фактичні частоти:")
print(observed_norm)

n = len(climate)

expected_proportions = [0.375, 0.25, 0.375]
expected_norm = [p * n for p in expected_proportions]

print("\nОчікувані частоти:")
print(expected_norm)

norm_result = chisquare(
    f_obs=observed_norm,
    f_exp=expected_norm
)

print(f"\nСтатистика chi2: {norm_result.statistic:.4f}")
print(f"p-value: {norm_result.pvalue:.6f}")

Фактичні частоти:
відхилення_від_норми
холодніше    20
звичайно      8
тепліше      20
Name: count, dtype: int64

Очікувані частоти:
[18.0, 12.0, 18.0]

Статистика chi2: 1.7778
p-value: 0.411112


In [19]:
alpha = 0.05

if norm_result.pvalue < alpha:
    print(
        "Висновок: H0 відхиляємо. "
        "Фактичний розподіл температурних категорій "
        "статистично відрізняється від заданих пропорцій."
    )
else:
    print(
        "Висновок: немає достатніх підстав відхиляти H0. "
        "Фактичний розподіл температурних категорій "
        "не має статистично значущої відмінності від заданих пропорцій."
    )

Висновок: немає достатніх підстав відхиляти H0. Фактичний розподіл температурних категорій не має статистично значущої відмінності від заданих пропорцій.


### Контролюючі питання

**1. Чому у критерії хі-квадрат використовуються квадрати відхилень і ділення на очікувану частоту?**

Квадрати відхилень роблять усі різниці додатними та показують величину розбіжності між фактичними й очікуваними частотами. Ділення на очікувану частоту враховує масштаб категорії. Чим більша відносна розбіжність, тим більший внесок клітинки у статистику χ².

**2. Як обчислюється очікувана частота для клітинки таблиці спряженості за умови незалежності ознак?**

Очікувана частота обчислюється за формулою:

$$
E_{ij} = \frac{R_i \cdot C_j}{N}
$$

де \(R_i\) — сума рядка, \(C_j\) — сума стовпця, а \(N\) — загальна кількість спостережень.

**3. Чим відрізняється твердження «велике p-value — недостатньо доказів відхилити H0» від «велике p-value — доводить незалежність»?**

Велике p-value означає, що отримані дані не дають достатніх підстав відхилити нульову гіпотезу. Однак це не є доказом того, що H0 безумовно істинна або що ознаки точно незалежні.

**4. Що робити, якщо очікувана частота в клітинці менша за 5?**

Якщо очікувана частота менша за 5, наближення χ² може бути ненадійним. Доцільно об'єднати близькі категорії, якщо це має змістовне обґрунтування, а потім повторно виконати критерій χ².


### Загальний висновок

У практичній роботі було досліджено критерій хі-квадрат та його застосування для аналізу таблиць спряженості й перевірки відповідності розподілу заданим пропорціям.

Було сформовано таблицю спряженості між сезоном і категорією відхилення температури від норми, обчислено очікувані частоти та виконано критерій незалежності χ². Також було перевірено умову щодо мінімальних очікуваних частот і продемонстровано об'єднання категорій.

За допомогою критерію χ² також перевірено рівномірність розподілу спостережень за сезонами та відповідність розподілу температурних категорій заданим пропорціям.

У результаті було закріплено навички роботи з функціями `chi2_contingency()` та `chisquare()`, побудови таблиць спряженості та інтерпретації статистики χ² і p-value.
